# Word2Vec
## Introducción
- Familia de modelos basados en redes neuronales superificales (dos capas)
- Estan dicesñados para procesar grandes cantidades de texto y aprender representaciones vectoriales de las palabras
## Word Embeddings
- **Word2Vec** asigna a cada palabra un vector en un espacio multidimensional.
- El algoritmo ordena los vectores generados de manera que las palabras que comparten contextos similares se agrupen cerca unas de otras
- La distancia entre las palabras se mide utilizando la **Similitud Coseno**: $\text{similitud}(A,B) = \frac{A \cdot B}{||A|| ||B||}$
## Hipotesis distribucional y contexto
- Teorema: ***"Conocerás cada palabra por las compañias que frecuenta"***
- El modelo no sabe que significan las palabras realmente, sino que aprende su significado al observar las palabras que aparecen a su alrededor.
## Context Window
- **Word2Vec** usa una ventana deslizante alrededor de la data de entrenamiento.
- Dicha ventana define basicamente cuantas palabras antes y despues se consideran como contexto de una palabra.



In [15]:
import numpy as np
from collections import Counter


class Word2VecSkipGram:
    def __init__(self, dim=10, ventana=2, lr=0.01):
        self.dim = dim
        self.ventana = ventana
        self.lr = lr

    def construir_vocabulario(self, corpus_tokens):
        conteo = Counter(t for sent in corpus_tokens for t in sent)
        self.vocab = sorted(conteo.keys())
        self.w2i = {w: i for i, w in enumerate(self.vocab)}
        self.i2w = {i: w for w, i in self.w2i.items()}
        self.V = len(self.vocab)

    def inicializar_pesos(self):
        self.W = np.random.uniform(-0.5, 0.5, (self.V, self.dim)) / self.dim
        self.W2 = np.random.uniform(-0.5, 0.5, (self.dim, self.V)) / self.dim

    def softmax(self, x):
        e = np.exp(x - np.max(x))
        return e / e.sum()

    def forward(self, centro_idx):
        h = self.W[centro_idx]
        z = self.W2.T @ h
        y = self.softmax(z)
        return h, z, y

    def backward(self, centro_idx, ctx_idx, h, y):
        e = y.copy()
        e[ctx_idx] -= 1
        dW2 = np.outer(h, e)
        dh = self.W2 @ e
        self.W2 -= self.lr * dW2
        self.W[centro_idx] -= self.lr * dh

    def entrenar(self, corpus_tokens, epocas=100):
        self.construir_vocabulario(corpus_tokens)
        self.inicializar_pesos()
        pares = []
        for sent in corpus_tokens:
            idx = [self.w2i[w] for w in sent if w in self.w2i]
            for i, centro in enumerate(idx):
                inicio = max(0, i - self.ventana)
                fin = min(len(idx), i + self.ventana + 1)
                for j in range(inicio, fin):
                    if j != i:
                        pares.append((centro, idx[j]))
        print(f"Total de pares de entrenamiento: {len(pares)}")
        for ep in range(epocas):
            np.random.shuffle(pares)
            perdida = 0
            for centro_idx, ctx_idx in pares:
                h, z, y = self.forward(centro_idx)
                perdida -= np.log(y[ctx_idx] + 1e-9)
                self.backward(centro_idx, ctx_idx, h, y)
            if (ep + 1) % 20 == 0:
                print(f"Época {ep + 1}/{epocas}  |  Pérdida: {perdida:.4f}")

    def embedding(self, palabra):
        return self.W[self.w2i[palabra]]

    def similares(self, palabra, n=5):
        vec = self.embedding(palabra)
        sims = []
        for w in self.vocab:
            if w == palabra:
                continue
            v2 = self.W[self.w2i[w]]
            cos = np.dot(vec, v2) / (np.linalg.norm(vec) * np.linalg.norm(v2) + 1e-9)
            sims.append((w, cos))
        return sorted(sims, key=lambda x: -x[1])[:n]

In [16]:
if __name__ == "__main__":
    corpus = [
        ["el", "perro", "come", "hueso"],
        ["el", "gato", "bebe", "leche"],
        ["el", "perro", "bebe", "agua"],
        ["el", "gato", "come", "pescado"],
        ["el", "pájaro", "come", "semillas"],
        ["el", "pájaro", "bebe", "agua"],
    ]

    modelo = Word2VecSkipGram(dim=10, ventana=2, lr=0.05)
    modelo.entrenar(corpus, epocas=200)

    print("\n── Vocabulario ──")
    print(modelo.vocab)

    print("\n── Embedding de 'perro' ──")
    print(np.round(modelo.embedding("perro"), 4))

    palabras_prueba = ["perro", "gato", "come", "bebe"]
    for palabra in palabras_prueba:
        print(f"\n── Palabras más similares a '{palabra}' ──")
        for w, score in modelo.similares(palabra, n=3):
            print(f"  {w:12s}  coseno={score:.4f}")

Total de pares de entrenamiento: 60
Época 20/200  |  Pérdida: 125.0383
Época 40/200  |  Pérdida: 110.3138
Época 60/200  |  Pérdida: 103.1380
Época 80/200  |  Pérdida: 101.5909
Época 100/200  |  Pérdida: 101.9760
Época 120/200  |  Pérdida: 101.4545
Época 140/200  |  Pérdida: 101.2664
Época 160/200  |  Pérdida: 101.1920
Época 180/200  |  Pérdida: 101.1284
Época 200/200  |  Pérdida: 100.8732

── Vocabulario ──
['agua', 'bebe', 'come', 'el', 'gato', 'hueso', 'leche', 'perro', 'pescado', 'pájaro', 'semillas']

── Embedding de 'perro' ──
[ 0.2826  0.2854 -1.3352 -0.3014  0.4972  0.2954 -0.0979 -1.1221  0.4221
  1.1609]

── Palabras más similares a 'perro' ──
  pájaro        coseno=0.5573
  gato          coseno=0.3657
  semillas      coseno=0.1128

── Palabras más similares a 'gato' ──
  pájaro        coseno=0.3702
  perro         coseno=0.3657
  hueso         coseno=0.2006

── Palabras más similares a 'come' ──
  bebe          coseno=0.3300
  agua          coseno=0.0817
  leche         cosen